<a href="https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel — never the _sample/final month for label logic

In [10]:
# Discover actual columns before writing anything else
con.sql(f"DESCRIBE SELECT * FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet' LIMIT 0")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Unit of analysis:**
One row = one pseudonymized content item's performance on one report date, for one pseudonymized client.
The grain is (report_date, client_id, content_id) — daily, not pre-aggregated per page.

**Time window:**
month=2026-03 (a mid-panel month, chosen deliberately). The final month in the warehouse
(the _sample table, June 2026) is held out and never used to develop label logic — it's the
natural outcome window for any past→future label, so touching it here would be leakage
before I've even started modeling.

**Reason:**
Daily grain is the actual grain fact_content_daily_performance ships at (confirmed by query
in part 3, not assumed) — aggregating up to "per page" without checking first would be a guess,
and this assignment is about proving claims with queries, not asserting them.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


| Category | Fields                                                                                  | Why |
|----------|------------------------------------------------------------------------------------------|-----|
| Features | clicks_7d_avg, impressions_7d_avg, ctr_trend, avg_position_delta, days_since_last_update | Each is computable from data available strictly before the decision date. |
| Label    | click_growth_flag (or similar future-window outcome — name it once you confirm it against DESCRIBE) | This is the thing being predicted, so it never enters the feature set. |
| Context  | report_date, client_id, content_id, month partition                                      | Identifies/positions a row but isn't itself predictive signal to feed the model. |
| Excluded | any future-window aggregate (e.g. next_30d_clicks, next_period CTR), and anything from fact_content_query_90d that overlaps the label's outcome window | These are only known *after* the decision moment — including them would be the leakage trap in part 4, not a legitimate feature. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
# (a) Grain: one row really is report_date × client × content
con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ distinct_keys │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │       9841378 │
└────────────┴───────────────┘

In [15]:
# (b) Slice size + date span
con.sql(f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet'
""")

┌─────────┬────────────┬────────────┐
│ n_rows  │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

In [16]:
# (c) Availability — filter with IS TRUE, show survival count
con.sql(f"""
DESCRIBE SELECT * FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet' LIMIT 0
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

**Verification claims:**

I prove three things about this slice, each with one query below:

1. **Grain** — `fact_content_daily_performance` for month=2026-03 is genuinely
   one row per (report_date, client_id, content_id): the total row count and the
   count of distinct (report_date, client_id, content_id) triples match exactly.
   No duplicate keys, no hidden aggregation.

2. **Slice size + date span** — the row count for this month and the min/max
   report_date confirm this is a full mid-panel month, not a partial or
   truncated partition, and not the sealed final month (June 2026 / `_sample`).

3. **Availability** — filtering with `IS TRUE` on the availability flag shows
   how many rows actually carry usable signal versus how many exist in the
   raw partition. The gap between total rows and surviving rows is itself a
   fact about this slice, not noise to average away.

Column names below are pinned to whatever `DESCRIBE` returned earlier in this
notebook — not assumed.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Named limitation:**

This slice cannot be compared cleanly across clients, because the panel is
unbalanced: `dim_clients.gsc_data_start` / `ga4_data_start` differ per client,
so some clients simply have no rows at all in earlier months while others have
a full history. A client with a "longer" March-to-present trend isn't
necessarily performing better — they may just have been onboarded earlier.
Any feature or comparison built across clients without accounting for this
will silently favor longer-tenured clients over newer ones.

**Other things this data can never tell you:**

- **Causation, not correlation.** A rise in clicks alongside a content update
  doesn't prove the update caused it — Google's ranking algorithm, seasonality,
  and competitor changes are all invisible here.
- **Window overlaps.** `fact_content_query_90d` is a fixed 90-day window with
  last-30/prev-30 sub-windows — if that window crosses into a period after the
  decision date being modeled, using it as a feature silently pulls in future
  information (the same trap named in part 2's Excluded row).
- **Pseudonymization ceiling.** Hash keys are salted and namespaced with no
  shipped salt — this data can never be traced back to a real client, domain,
  or query, which is by design but also means no external enrichment
  (industry, geography, real domain authority) is possible from this table alone.

Claims from this slice should be framed as observed / measured / directional /
decision-support — never as a claim about causation or about Google's actual
ranking algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.